# Synthetic data engineering

### $$\Delta S_t = \theta (\mu - S_{t-1}) + \sigma_{R_t} \epsilon_t + J_t$$

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('../scripts'))

# Import your pipeline
from synthetic import SYNTHETIC
from spread import SPREAD
from descriptive import DESCRIPTIVE

### Define ground truth parameters

### Generate the paths

In [ ]:
# Instantiate the generator (simulating ~1 month of heavy tick data)
generator = SYNTHETIC(n_ticks=250000, start_date="2026-01-01", symbol_a="SYN_A", symbol_b="SYN_B")

# Generate the hidden math
true_df = generator.generate_market()

# Plot the hidden states to prove our math works
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(true_df['datetime'], true_df['true_spread'], color='blue', alpha=0.6, label='True Spread')
ax2 = ax1.twinx()
ax2.fill_between(true_df['datetime'], 0, true_df['true_state'], color='red', alpha=0.2, label='Hidden Danger Regime')
ax1.set_title("Ground Truth: Mean-Reverting Spread with Volatility Regimes & Poisson Jumps")
plt.show()

### Save to parquet

In [ ]:
# Apply Bid/Ask bounces and save to disk
synthetic_files = generator.save_to_parquets(output_dir="../data/synthetic")

# Let's look at what we created
test_read = pd.read_parquet(synthetic_files[0])
print("\nParquet Schema Check:")
display(test_read.head())

### Run the sanity check

In [ ]:
# If your SPREAD class accepts this without crashing, your sandbox is perfectly integrated
builder = SPREAD(agg_type='volume', threshold=1000, active_hours=(0, 24)) 
df_synth = builder.build(synthetic_files)

# Use your existing diagnostics!
builder.plot_diagnostics()

### Visual proof

In [ ]:
# Run your Descriptive Stats. 
# We WANT to see high Kurtosis here. If we do, it means our synthetic jumps 
# successfully mimic the real-world anomalies that break the Markov model.
eda = DESCRIPTIVE(df_synth, name_a="SYN_A", name_b="SYN_B")
eda.generate_full_eda()